# 📘 Day 34 — Ensemble Methods: Voting & Bagging

**Author:** Sahil-K-Y 
**Phase:** 3 — Tree Models & SVM 
**Date:** Day 034 of 214-Day AI/ML Roadmap

---
## 📌 Overview

Ensemble methods combine **multiple models** to produce a **stronger learner**. Today we cover the two foundational ensemble techniques: **Voting** and **Bagging**.

### 🎯 Learning Objectives
- Understand the **wisdom of crowds** principle in ML
- Implement **Hard Voting** (majority vote) and **Soft Voting** (probability average)
- Master **BaggingClassifier** with bootstrap sampling
- Understand **Out-of-Bag (OOB) score** as free validation
- Compare ensemble performance vs individual models

---
## 🧠 1. Why Ensembles Work

### The Wisdom of Crowds
If each classifier has accuracy slightly better than random (>50%), combining them **reduces error exponentially**.

### Bias-Variance Tradeoff
| Technique | Reduces | How |
|-----------|---------|-----|
| **Bagging** | Variance | Averages out noisy predictions |
| **Boosting** | Bias | Focuses on hard examples (Day 39+) |
| **Stacking** | Both | Meta-learner combines diverse models |

### Key Condition: Diversity
Ensembles work best when base models are **diverse** — they make **different errors** on different samples.

---
## 🧠 2. Voting Classifier

### 2.1 Hard Voting (Majority Vote)
Each model votes for a class → the class with **most votes wins**.

```
Model 1: Class A    ┐
Model 2: Class B    ├── Majority Vote → Class A (2 vs 1)
Model 3: Class A    ┘
```

### 2.2 Soft Voting (Probability Average)
Each model outputs **class probabilities** → average probabilities → pick the highest.

```
Model 1: [0.7, 0.3]    ┐
Model 2: [0.4, 0.6]    ├── Average: [0.60, 0.40] → Class A
Model 3: [0.7, 0.3]    ┘
```

### Soft vs Hard: Which is Better?
- **Soft voting** is usually better because it considers **confidence levels**
- Requires all models to support `predict_proba()` (set `probability=True` for SVC)
- A highly confident minority can outweigh an uncertain majority

---
## 🧠 3. Bagging (Bootstrap Aggregating)

### How Bagging Works:
1. Create **B bootstrap samples** (random sampling **with replacement**) from training data
2. Train **one model per bootstrap sample**
3. **Aggregate predictions** (majority vote for classification, mean for regression)

### Bootstrap Sampling:
- Each bootstrap sample has **same size as original** but ~63.2% unique samples
- The remaining ~36.8% are called **Out-of-Bag (OOB)** samples

### Why Bagging Reduces Variance:
- Individual trees might overfit to their specific bootstrap sample
- When averaged, the **noise cancels out**, leaving the signal
- Works best with **high-variance** models (deep trees, KNN with small k)

### ⚠️ Bagging Does NOT Help When:
- Base model has **high bias** (underfitting) — e.g., linear regression on non-linear data
- All base models make the **same errors** (no diversity)

---
## 🧠 4. Out-of-Bag (OOB) Score

### What is OOB?
Each bootstrap sample leaves out ~36.8% of data. These **out-of-bag samples** serve as a **free validation set**.

### How OOB Works:
1. For each training sample, identify which base models did **NOT** train on it
2. Use only those models to predict that sample
3. Average these predictions to get the **OOB score**

### Why OOB is Useful:
- **Free validation** without needing a separate val set
- Similar accuracy to **cross-validation** but computationally cheaper
- Enable with `BaggingClassifier(oob_score=True)`

### ⚠️ Note:
OOB requires `bootstrap=True` (default). Won't work with `bootstrap=False`.

---
## 💻 Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.ensemble import VotingClassifier, BaggingClassifier
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
sns.set_palette('Set2')
print('✅ All libraries imported successfully!')

---
## 📊 Dataset: Satellite Image Classification (Statlog)

The **Statlog Satellite** dataset contains **6435 samples** of multi-spectral satellite image data, classified into **6 soil/land types** based on 36 features (pixel intensities from 4 spectral bands across 3x3 neighborhoods).

**6435 samples, 36 features, 6 classes**

**Why this dataset?** High-dimensional multi-class problem where individual models struggle but **ensembles can significantly boost performance**.

In [ ]:
# --- Load Satellite Image Dataset ---
satellite = fetch_openml('satellite', version=1, as_frame=True, parser='auto')
df = satellite.frame

X = df.drop(columns=[satellite.target_names[0]]).astype(float)
y = LabelEncoder().fit_transform(df[satellite.target_names[0]])

print(f'Shape: {X.shape}')
print(f'Number of classes: {len(np.unique(y))}')
print(f'Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}')
print(f'\nFeature stats:')
X.describe().loc[['mean', 'std', 'min', 'max']].T.head(10)

---
## ✏️ Exercise 1: Individual Model Baselines
Train individual classifiers as baselines.

**Tasks:**
1. Split data 80/20 stratified
2. Train: LR, DT, SVM(rbf), KNN, NB individually
3. Record 5-fold CV accuracy for each
4. Display baseline leaderboard

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 2: Hard Voting vs Soft Voting
Build ensemble VotingClassifiers.

**Tasks:**
1. Create VotingClassifier with hard voting (all 5 models)
2. Create VotingClassifier with soft voting
3. Compare ensemble accuracy vs best individual model
4. Show how much improvement the ensemble provides

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 3: BaggingClassifier with Decision Trees
Apply Bagging with different n_estimators.

**Tasks:**
1. Train BaggingClassifier with n_estimators = [5, 10, 25, 50, 100]
2. Use DecisionTreeClassifier as base estimator
3. Enable oob_score=True
4. Plot n_estimators vs accuracy AND oob_score
5. Find the sweet spot (diminishing returns)

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 4: Bagging with Different Base Estimators
Test bagging with different base models.

**Tasks:**
1. Bagging with DecisionTree (default)
2. Bagging with KNN
3. Bagging with SVM
4. Compare — which base estimator benefits most from bagging?
5. Discuss why (hint: bias vs variance)

In [ ]:
# YOUR CODE HERE


---
## ✏️ Exercise 5: Bagging Hyperparameter Tuning
Tune max_samples and max_features.

**Tasks:**
1. Test max_samples = [0.5, 0.7, 0.8, 1.0]
2. Test max_features = [0.5, 0.7, 0.8, 1.0]
3. Create a heatmap of accuracy for each combination
4. Find the optimal settings

In [ ]:
# YOUR CODE HERE


---
## 📝 Key Takeaways
1. Hard vs Soft voting winner: ...
2. Ensemble vs best individual improvement: ...
3. Optimal n_estimators for bagging: ...
4. Best base estimator for bagging: ...